In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
# Assuming maxence.ipynb is in REXIA/Project/ and data is in REXIA/Project/data/
try:
    df = pd.read_csv("data/RH_dataset.csv", sep=";")
except FileNotFoundError:
    # Fallback if maxence.ipynb is in a notebooks/ folder
    df = pd.read_csv("../data/RH_dataset.csv", sep=";")

df.head()

,Famille d'emploi,Dernière promotion (mois),Dernière augmentation (mois),Début de contrat (années),Ancienneté groupe (années),Etablissement,Âge (années),Parent,Niveau hiérarchique,Salaire (Euros),Statut marital,Véhicule,matricule,label
0,Production,8.510000,7.900000,0.910000,0.970000,27,30,1,1,3199,Marié(e),0,32,0
1,Production,35.119999,22.690001,14.830000,16.299999,7,45,1,2,3861,Marié(e),1,1890,0
2,Production,25.299999,22.139999,17.309999,17.790001,28,49,1,2,4324,PACS,1,1847,0
3,Production,5.240000,5.100000,1.020000,1.750000,27,24,0,1,2641,Célibataire,0,2619,1
4,Production,35.919998,22.840000,8.050000,9.000000,7,46,1,2,5072,Marié(e),1,1963,0


In [2]:
# 1. Vérifier le nombre moyen de lignes par personne (matricule)
lignes_par_matricule = df['matricule'].value_counts()
print(f"Nombre moyen de lignes par matricule : {lignes_par_matricule.mean():.2f}")
print(f"Nombre max : {lignes_par_matricule.max()}, Nombre min : {lignes_par_matricule.min()}\n")

# 2. Vérifier si les lignes pour une même personne sont uniques (pas de vrais doublons parfaits)
nb_total_lignes = len(df)
nb_lignes_uniques = len(df.drop_duplicates())
print(f"Il y a {nb_total_lignes} lignes au total, dont {nb_lignes_uniques} uniques.")
if nb_total_lignes == nb_lignes_uniques:
    print("=> Aucune ligne n'est un doublon parfait. Chaque ligne d'une même personne diffère sur au moins une caractéristique.\n")

# 3. Vérifier la dynamique (par exemple s'il s'agit d'une observation par an)
# On va trier par 'matricule' et par 'Ancienneté groupe (années)' ou 'Âge (années)'
cols_temps = [' matricule ', 'Ancienneté groupe (années)', 'Âge (années)']
df_sort = df.sort_values(by=['matricule', 'Ancienneté groupe (années)', 'Âge (années)'])

# Regardons l'écart type de la différence d'âge ou d'ancienneté d'une ligne à l'autre pour une même personne
df_sort['Diff_Age'] = df_sort.groupby('matricule')['Âge (années)'].diff()
df_sort['Diff_Anciennete'] = df_sort.groupby('matricule')['Ancienneté groupe (années)'].diff()

print("Différence d'âge entre deux lignes d'une même personne (statistiques basiques) :")
print(df_sort['Diff_Age'].value_counts(dropna=True).head())

print("\nDifférence d'ancienneté entre deux lignes d'une même personne (statistiques basiques) :")
print(df_sort['Diff_Anciennete'].value_counts(dropna=True).head())

# Exemples
print("\nExemple des trajectoires pour un matricule ayant plusieurs lignes :")
exemple_matricule = lignes_par_matricule.idxmax()
display(df_sort[df_sort['matricule'] == exemple_matricule].head(10))

Nombre moyen de lignes par matricule : 8.92
Nombre max : 15, Nombre min : 1

Il y a 23857 lignes au total, dont 23857 uniques.
=> Aucune ligne n'est un doublon parfait. Chaque ligne d'une même personne diffère sur au moins une caractéristique.

Différence d'âge entre deux lignes d'une même personne (statistiques basiques) :
Diff_Age
1.0    11699
0.0     9298
2.0      126
3.0       51
4.0        5
Name: count, dtype: int64

Différence d'ancienneté entre deux lignes d'une même personne (statistiques basiques) :
Diff_Anciennete
0.300000    1558
0.299999    1002
0.300001     406
0.300000     378
0.300000     366
Name: count, dtype: int64

Exemple des trajectoires pour un matricule ayant plusieurs lignes :


,Famille d'emploi,Dernière promotion (mois),Dernière augmentation (mois),Début de contrat (années),Ancienneté groupe (années),Etablissement,Âge (années),Parent,Niveau hiérarchique,Salaire (Euros),Statut marital,Véhicule,matricule,label,Diff_Age,Diff_Anciennete
0,Production,8.510000,7.900000,0.91,0.97,27,30,1,1,3199,Marié(e),0,32,0,NaN,NaN
435,Production,11.280000,10.750000,1.18,1.20,27,30,1,1,3200,Marié(e),0,32,0,0.0,0.23
1735,Production,14.040000,13.590000,1.45,1.45,27,31,1,1,3200,Marié(e),0,32,0,1.0,0.25
3247,Production,16.920000,16.370001,1.70,1.70,27,32,1,1,3199,Marié(e),0,32,0,1.0,0.25
4799,Production,19.889999,19.340000,2.08,2.08,28,33,1,1,3199,Marié(e),0,32,0,1.0,0.38
6358,Etudes & Technique,2.350000,21.889999,2.27,2.38,27,34,1,1,3202,Marié(e),0,32,0,1.0,0.30
7967,Etudes & Technique,5.960000,0.960000,2.53,2.53,27,34,1,1,3560,Marié(e),0,32,0,0.0,0.15
9637,Etudes & Technique,8.710000,3.890000,2.77,2.77,27,34,1,1,3559,Marié(e),0,32,0,0.0,0.24
11353,Etudes & Technique,11.500000,6.770000,3.03,3.03,27,34,1,1,3560,Marié(e),0,32,0,0.0,0.26
13091,Etudes & Technique,14.770000,1.980000,3.25,3.31,27,35,1,1,3773,Marié(e),0,32,0,1.0,0.28


In [3]:
# Analysons ce qui change précisément d'une ligne à l'autre pour une personne qui a beaucoup de lignes
df_person = df_sort[df_sort['matricule'] == exemple_matricule].copy()

# Regarder quelles colonnes varient
cols_variables = []
cols_fixes = []

for col in df_person.columns:
    if col not in ['matricule', 'Diff_Age', 'Diff_Anciennete'] and df_person[col].nunique() > 1:
        cols_variables.append(col)
    elif col not in ['matricule', 'Diff_Age', 'Diff_Anciennete']:
        cols_fixes.append(col)

print("Colonnes fixes pour cette personne :", cols_fixes)
print("Colonnes qui changent :", cols_variables)

df_person[cols_variables + ['Ancienneté groupe (années)']]

Colonnes fixes pour cette personne : ['Parent', 'Statut marital', 'label']
Colonnes qui changent : ["Famille d'emploi", 'Dernière promotion (mois)', 'Dernière augmentation (mois)', 'Début de contrat (années)', 'Ancienneté groupe (années)', 'Etablissement', 'Âge (années)', 'Niveau hiérarchique', 'Salaire (Euros)', 'Véhicule']


,Famille d'emploi,Dernière promotion (mois),Dernière augmentation (mois),Début de contrat (années),Ancienneté groupe (années),Etablissement,Âge (années),Niveau hiérarchique,Salaire (Euros),Véhicule,Ancienneté groupe (années)
0,Production,8.510000,7.900000,0.91,0.97,27,30,1,3199,0,0.97
435,Production,11.280000,10.750000,1.18,1.20,27,30,1,3200,0,1.20
1735,Production,14.040000,13.590000,1.45,1.45,27,31,1,3200,0,1.45
3247,Production,16.920000,16.370001,1.70,1.70,27,32,1,3199,0,1.70
4799,Production,19.889999,19.340000,2.08,2.08,28,33,1,3199,0,2.08
6358,Etudes & Technique,2.350000,21.889999,2.27,2.38,27,34,1,3202,0,2.38
7967,Etudes & Technique,5.960000,0.960000,2.53,2.53,27,34,1,3560,0,2.53
9637,Etudes & Technique,8.710000,3.890000,2.77,2.77,27,34,1,3559,0,2.77
11353,Etudes & Technique,11.500000,6.770000,3.03,3.03,27,34,1,3560,0,3.03
13091,Etudes & Technique,14.770000,1.980000,3.25,3.31,27,35,1,3773,0,3.31


### Conclusion de l'analyse longitudinale

- **Multitude d'entrées par personne** : En moyenne, il y a bien 8.9 lignes statistiques pour le même `matricule`. Les personnes peuvent avoir jusqu'à 15 lignes (et au minimum 1).
- **Unicité parfaite des lignes** : Il n'y a **aucun doublon exact**. Chaque ligne capture un moment différent de la vie du collaborateur. 
- **La nature des données** : Le jeu de données fonctionne comme un **historique de l'employé** (données longitudinales). La différence de l'ancienneté d'une ligne à l'autre suggère que le logiciel enregistre une "photo" de chaque individu à différents moments du temps (approximativement tous les ~0.3 an, ou lors de moments clés ou entretiens d'évaluation). 
- **Évolution dans le temps** : D'une ligne à l'autre, on voit évoluer progressivement : _l'âge, le temps passé depuis la dernière promotion/augmentation, l'ancienneté._ Mais on observe aussi des **sauts qualitatifs** : _changement d'établissement, augmentation de salaire, passage d'un niveau hiérarchique supérieur, acquisition d'un véhicule, etc._

Si vous souhaitez entraîner un modèle sur ces données (par exemple pour prédire une démission dans l'année qui vient), il faudra être très vigilant : on ne peut pas traiter chaque ligne comme un individu indépendant au risque de causer une "fuite de données" (Data Leakage) et de biaiser artificiellement l'apprentissage (un même matricule ne doit pas être présent en même temps dans le jeu d'entraînement et de test).